In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ── 0. Install dependencies ─────────────────────────────────────────────────
!pip install -q -U transformers peft bitsandbytes accelerate datasets "pillow<12.0" safetensors torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 148.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 116.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 50.4 MB/s eta 0:00:00


In [3]:
import os
os.chdir("/content/drive/MyDrive/DL FInal")

In [4]:
# ── 1. Imports and global config ─────────────────────────────────────────────
import os, json, math, random
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from tqdm.auto import tqdm

from transformers import AutoProcessor, BitsAndBytesConfig, TrainingArguments, Trainer

try:
    from transformers import AutoModelForVision2Seq as SmolVLMModelClass
except ImportError:
    from transformers import AutoModelForImageTextToText as SmolVLMModelClass

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# Change this to your competition data folder.
DATA_DIR = Path('Pixels to Predictions')
OUTPUT_DIR = Path("p2p_candidate_yesno_outputs_train_val")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = DATA_DIR / "train.csv"
VAL_CSV = DATA_DIR / "val.csv"
TEST_CSV = DATA_DIR / "test.csv"

IMAGE_LONGEST_EDGE = 384
IMAGE_SEQ_LEN = 64

MAX_LECTURE_CHARS = 0
MAX_HINT_CHARS = 0
USE_METADATA = False

TRAIN_ON_VAL_TOO = True  # Set True only for final submission training after validation is healthy.

EPOCHS = 1
TRAIN_BATCH_SIZE = 4
GRAD_ACCUM = 1
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
LOGGING_STEPS = 20
SAVE_STEPS = 500
MAX_GRAD_NORM = 1.0

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
USE_DORA = False
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj","out_proj"]

USE_4BIT = False
USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
USE_FP16 = torch.cuda.is_available() and not USE_BF16

EVAL_BATCH_SIZE = 32

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16:", USE_BF16, "fp16:", USE_FP16)

CUDA: True
GPU: NVIDIA A100-SXM4-80GB
bf16: True fp16: False


In [5]:
# ── 2. Load data ─────────────────────────────────────────────────────────────
def parse_choices(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        try:
            out = json.loads(x)
            if isinstance(out, list):
                return out
        except Exception:
            pass
        return [p.strip().strip("'\"") for p in x.strip("[]").split(",") if p.strip()]
    return list(x)

def load_split(csv_path, split, has_answer=True):
    df = pd.read_csv(csv_path)
    df["split"] = split
    df["choices"] = df["choices"].apply(parse_choices)
    df["num_choices"] = df["num_choices"].astype(int)
    if has_answer:
        df["answer"] = df["answer"].astype(int)
    return df

train_df = load_split(TRAIN_CSV, "train", has_answer=True)
val_df = load_split(VAL_CSV, "val", has_answer=True)
test_df = load_split(TEST_CSV, "test", has_answer=False)

print(train_df.shape, val_df.shape, test_df.shape)
train_df.head()

(3109, 16) (1048, 16) (1008, 14)


,id,image_path,question,choices,num_choices,answer,hint,lecture,solution,task,grade,subject,topic,category,skill,split
0,train_07667,images/train/train_07667.png,Why might putting each tadpole in its own pool...,[the male's tadpoles will be larger when they ...,3,2,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...,train
1,train_02628,images/train/train_02628.png,Why might forming strong social bonds with oth...,"[the female's offspring will live longer, the ...",3,0,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...,train
2,train_00927,images/train/train_00927.png,Why might raising cubs with other lionesses in...,"[the lioness's cubs will be around other cubs,...",3,1,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...,train
3,train_10389,images/train/train_10389.png,Why might removing broken eggshells from the n...,"[the gull's chicks will get food, the gull's o...",3,1,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...,train
4,train_09024,images/train/train_09024.png,Why might feeding offspring during mouthbroodi...,"[the female will become weak and unhealthy, th...",3,1,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...,train


In [6]:
# ── 3. Path and prompt helpers ───────────────────────────────────────────────
def _clean_text(x, max_chars=None):
    x = "" if x is None or pd.isna(x) else str(x).strip()
    if max_chars is not None and len(x) > max_chars:
        x = x[:max_chars].rsplit(" ", 1)[0] + " ..."
    return x

def resolve_image_path(row):
    raw = Path(str(row["image_path"]))
    if raw.is_absolute() and raw.exists():
        return raw

    candidates = [
        DATA_DIR / raw,
        DATA_DIR / "images" / "images"  / str(row["split"]) / raw.name,
        DATA_DIR / str(row["split"]) / raw.name,
    ]

    for p in candidates:
        if p.exists():
            return p

    raise FileNotFoundError(f"Could not resolve image path for id={row.get('id')}: {row['image_path']}")

def load_and_transform_image(row):
    """Load image and apply the pipeline's stretch-resize to 384x384."""
    path = resolve_image_path(row)
    image = Image.open(path).convert("RGB")
    image = image.resize((IMAGE_LONGEST_EDGE, IMAGE_LONGEST_EDGE), Image.BICUBIC)
    return image

def build_base_question_prompt(row):
    """Match pipeline's PromptFormatter.render_prompt for candidate_yes_no."""
    parts = ["<image>"]
    parts.append(f"Question:\n{row['question']}")
    return "\n\n".join(parts)
def build_candidate_prompt_from_row(row, choice_idx, include_answer=False):
    base = build_base_question_prompt(row)
    choice_text = row["choices"][choice_idx]
    prompt = (
        f"{base}\n\n"
        f"Candidate answer:\n{choice_text}\n\n"
        f"Is the candidate answer correct? Reply with Yes or No only.\n"
        f"Answer:"
    )
    if include_answer:
        target = " Yes" if int(choice_idx) == int(row["answer"]) else " No"
        prompt += target
    return prompt


In [8]:
# ── 4. Build candidate yes/no training dataframe ─────────────────────────────
def make_candidate_df(df, has_answer=True):
    rows = []
    for _, row in df.iterrows():
        for choice_idx in range(int(row["num_choices"])):
            new_row = row.to_dict()
            new_row["candidate_idx"] = int(choice_idx)
            new_row["candidate_text"] = row["choices"][choice_idx]
            if has_answer:
                new_row["target"] = "Yes" if int(choice_idx) == int(row["answer"]) else "No"
                new_row["target_is_yes"] = int(choice_idx == int(row["answer"]))
                new_row["original_answer"] = int(row["answer"])
            rows.append(new_row)
    return pd.DataFrame(rows)

fit_source_df = pd.concat([train_df, val_df], ignore_index=True) if TRAIN_ON_VAL_TOO else train_df.copy()
train_candidate_df = make_candidate_df(fit_source_df, has_answer=True)

print("Original training questions:", len(fit_source_df))
print("Candidate training rows:", len(train_candidate_df))
print(train_candidate_df[["id", "candidate_idx", "target", "original_answer"]].head())
print("\nQuestion-level answer distribution:")
print(fit_source_df["answer"].value_counts(normalize=True).sort_index())

Original training questions: 4157
Candidate training rows: 12906
            id  candidate_idx target  original_answer
0  train_07667              0     No                2
1  train_07667              1     No                2
2  train_07667              2    Yes                2
3  train_02628              0    Yes                0
4  train_02628              1     No                0

Question-level answer distribution:
answer
0    0.353861
1    0.336781
2    0.236709
3    0.067116
4    0.005533
Name: proportion, dtype: float64


In [9]:
# ── 5. Datasets ──────────────────────────────────────────────────────────────
class CandidateYesNoTrainDataset(Dataset):
    def __init__(self, candidate_df):
        self.df = candidate_df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = load_and_transform_image(row)
        prompt_text = build_candidate_prompt_from_row(row, int(row["candidate_idx"]), include_answer=False)
        target = " Yes" if int(row["candidate_idx"]) == int(row["original_answer"]) else " No"
        return {
            "image": image,
            "prompt_text": prompt_text,
            "target_text": target,
            "id": row["id"],
            "candidate_idx": int(row["candidate_idx"]),
            "target": row["target"],
            "original_answer": int(row["original_answer"]),
        }

class MultipleChoiceEvalDataset(Dataset):
    def __init__(self, df, has_answer=True):
        self.df = df.reset_index(drop=True)
        self.has_answer = has_answer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = load_and_transform_image(row)
        item = {
            "id": row["id"],
            "image": image,
            "row": row.to_dict(),
            "choices": row["choices"],
            "num_choices": int(row["num_choices"]),
        }
        if self.has_answer:
            item["answer"] = int(row["answer"])
        return item

train_ds = CandidateYesNoTrainDataset(train_candidate_df)
val_eval_ds = MultipleChoiceEvalDataset(val_df, has_answer=True)
test_eval_ds = MultipleChoiceEvalDataset(test_df, has_answer=False)

print("train candidates:", len(train_ds))
print("val questions:", len(val_eval_ds))
print("test questions:", len(test_eval_ds))

train candidates: 12906
val questions: 1048
test questions: 1008


In [10]:
# ── 6. Processor and model ──────────────────────────────────────────────────
processor = AutoProcessor.from_pretrained(MODEL_ID, image_seq_len=IMAGE_SEQ_LEN)

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

print("processor.image_seq_len:", processor.image_seq_len)
print("image_processor.do_image_splitting:", processor.image_processor.do_image_splitting)
print("image_processor.size:", processor.image_processor.size)
print("image_processor.max_image_size:", processor.image_processor.max_image_size)

compute_dtype = torch.bfloat16 if USE_BF16 else (torch.float16 if USE_FP16 else torch.float32)

model = SmolVLMModelClass.from_pretrained(
    MODEL_ID,
    torch_dtype=compute_dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
)

model.config.use_cache = False
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGETS,
    use_dora=USE_DORA,
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable_params:,}")
assert trainable_params < 5_000_000, "Trainable parameter count exceeds 5M competition limit."

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

trainable params: 4,456,448 || all params: 511,938,752 || trainable%: 0.8705
Trainable params: 4,456,448


In [11]:
# ── 7. Answer-only collator ─────────────────────────────────────────────────
@dataclass
class CandidateAnswerOnlyCollator:
    processor: AutoProcessor

    def _encode(self, texts, images):
        kwargs = {
            "text": texts,
            "padding": True,
            "return_tensors": "pt",
        }
        if images and any(img is not None for img in images):
            kwargs["images"] = [[img] for img in images]
        return self.processor(**kwargs)

    def __call__(self, batch):
        prompt_texts = [x["prompt_text"] for x in batch]
        target_texts = [x["target_text"] for x in batch]
        full_texts = [p + t for p, t in zip(prompt_texts, target_texts)]
        images = [x["image"] for x in batch]

        full_batch = self._encode(full_texts, images)
        prompt_batch = self._encode(prompt_texts, images)

        labels = full_batch["input_ids"].clone()
        labels[full_batch["attention_mask"] == 0] = -100

        prompt_lengths = prompt_batch["attention_mask"].sum(dim=1)
        for i, prompt_len in enumerate(prompt_lengths.tolist()):
            labels[i, :prompt_len] = -100

        full_batch["labels"] = labels
        return full_batch

train_collator = CandidateAnswerOnlyCollator(processor=processor)

sample_batch = [train_ds[0], train_ds[1]]
batch_out = train_collator(sample_batch)
for k, v in batch_out.items():
    if torch.is_tensor(v):
        print(k, v.shape, v.dtype)

tmp = {k: (v.to(model.device) if torch.is_tensor(v) else v) for k, v in batch_out.items()}
with torch.no_grad():
    out = model(**tmp)
print("smoke loss:", float(out.loss))

pixel_values torch.Size([2, 17, 3, 512, 512]) torch.float32
pixel_attention_mask torch.Size([2, 17, 512, 512]) torch.int64
input_ids torch.Size([2, 1227]) torch.int64
attention_mask torch.Size([2, 1227]) torch.int64
labels torch.Size([2, 1227]) torch.int64
smoke loss: 4.141719818115234


In [12]:
# ── 8. Balanced answer-index sampler ────────────────────────────────────────
def make_balanced_answer_sampler(candidate_df):
    original_answer = candidate_df["original_answer"].astype(int).values
    counts = pd.Series(original_answer).value_counts().to_dict()
    weights = np.array([1.0 / counts[a] for a in original_answer], dtype=np.float64)
    return WeightedRandomSampler(
        weights=torch.DoubleTensor(weights),
        num_samples=len(weights),
        replacement=True,
    )

balanced_sampler = make_balanced_answer_sampler(train_candidate_df)

# Sanity check the sampled question-answer index distribution.
debug_weights = []
counts = train_candidate_df["original_answer"].value_counts().to_dict()
for a in train_candidate_df["original_answer"].astype(int).values:
    debug_weights.append(1.0 / counts[a])

sampled_indices = list(iter(WeightedRandomSampler(
    weights=torch.DoubleTensor(debug_weights),
    num_samples=min(2000, len(train_candidate_df)),
    replacement=True,
)))
sampled_answers = train_candidate_df.iloc[sampled_indices]["original_answer"].value_counts(normalize=True).sort_index()
print(sampled_answers)

original_answer
0    0.2090
1    0.1945
2    0.1925
3    0.1960
4    0.2080
Name: proportion, dtype: float64


In [13]:
# ── 9. Train ────────────────────────────────────────────────────────────────
from transformers.utils import logging
from datasets.utils.logging import enable_progress_bar

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    logging_steps=LOGGING_STEPS,
    gradient_checkpointing=False,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    tf32=True,
    bf16=USE_BF16,
    fp16=USE_FP16,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    remove_unused_columns=False,
    report_to="none",
    lr_scheduler_type="linear",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=train_collator,
)

def _get_train_dataloader_with_balanced_sampler():
    return DataLoader(
        train_ds,
        batch_size=training_args.per_device_train_batch_size,
        sampler=balanced_sampler,
        collate_fn=train_collator,
        num_workers=training_args.dataloader_num_workers,
        pin_memory=True,
    )

trainer.get_train_dataloader = _get_train_dataloader_with_balanced_sampler

trainer.train()

final_adapter_dir = OUTPUT_DIR / "final_adapter"
trainer.save_model(str(final_adapter_dir))
processor.save_pretrained(str(final_adapter_dir))
print("Saved adapter to:", final_adapter_dir)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
20,2.291510
40,1.782194
60,1.385216
80,0.897176
100,0.615955
120,0.678181
140,0.643793
160,0.690610
180,0.563153
200,0.705593


Saved adapter to: p2p_candidate_yesno_outputs_train_val/final_adapter


In [ ]:
# 9 minutes

In [10]:
# ── 10. Optional: load saved adapter for inference ───────────────────────────
LOAD_ADAPTER = True
ADAPTER_DIR = OUTPUT_DIR / "final_adapter"

if LOAD_ADAPTER:
    processor = AutoProcessor.from_pretrained(ADAPTER_DIR, image_seq_len=IMAGE_SEQ_LEN)
    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

    compute_dtype = torch.bfloat16 if USE_BF16 else (torch.float16 if USE_FP16 else torch.float32)

    base_model = SmolVLMModelClass.from_pretrained(
        MODEL_ID,
        torch_dtype=compute_dtype,
        device_map="auto" if torch.cuda.is_available() else None,
        low_cpu_mem_usage=True,
    )

    model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
    model.eval()
    print("Loaded adapter from:", ADAPTER_DIR)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Loaded adapter from: p2p_candidate_yesno_outputs_train_val/final_adapter


In [11]:
# # ── 11. Candidate yes/no batched evaluation ─────────────────────────────────
YES_TOKEN_IDS = processor.tokenizer.encode(" Yes", add_special_tokens=False)
NO_TOKEN_IDS = processor.tokenizer.encode(" No", add_special_tokens=False)

print("YES:", YES_TOKEN_IDS, processor.tokenizer.decode(YES_TOKEN_IDS))
print("NO:", NO_TOKEN_IDS, processor.tokenizer.decode(NO_TOKEN_IDS))

assert len(YES_TOKEN_IDS) == 1, YES_TOKEN_IDS
assert len(NO_TOKEN_IDS) == 1, NO_TOKEN_IDS

YES_TOKEN_ID = YES_TOKEN_IDS[0]
NO_TOKEN_ID = NO_TOKEN_IDS[0]

def build_candidate_prompt_for_eval(item, choice_idx):
    row = dict(item["row"])
    row["choices"] = item["choices"]
    row["num_choices"] = item["num_choices"]
    base = build_base_question_prompt(row)
    choice_text = item["choices"][choice_idx]
    return (
        f"{base}\n\n"
        f"Candidate answer:\n{choice_text}\n\n"
        f"Is the candidate answer correct? Reply with Yes or No only.\n"
        f"Answer:"
    )

@torch.inference_mode()
def score_candidates_yes_fast(model, processor, batch_items, use_yes_minus_no=False):
    model.eval()
    texts = []
    images = []
    meta = []
    for item_idx, item in enumerate(batch_items):
        for choice_idx in range(int(item["num_choices"])):
            prompt = build_candidate_prompt_for_eval(item, choice_idx)
            texts.append(prompt)
            images.append(item["image"])
            meta.append((item_idx, choice_idx))
    inputs = processor(
        text=texts,
        images=images,
        padding=True,
        return_tensors="pt",
    )
    inputs = {
        k: (v.to(model.device, non_blocking=True) if torch.is_tensor(v) else v)
        for k, v in inputs.items()
    }
    outputs = model(**inputs)
    last_positions = inputs["attention_mask"].sum(dim=1) - 1
    batch_idx = torch.arange(outputs.logits.size(0), device=outputs.logits.device)
    next_token_logits = outputs.logits[batch_idx, last_positions, :]
    log_probs = F.log_softmax(next_token_logits, dim=-1)
    if use_yes_minus_no:
        scores_flat = (log_probs[:, YES_TOKEN_ID] - log_probs[:, NO_TOKEN_ID]).detach().cpu().tolist()
    else:
        scores_flat = log_probs[:, YES_TOKEN_ID].detach().cpu().tolist()
    grouped_scores = [[] for _ in batch_items]
    for score, (item_idx, choice_idx) in zip(scores_flat, meta):
        grouped_scores[item_idx].append(score)
    return grouped_scores



YES: [9230]  Yes
NO: [2838]  No


In [12]:
def predict_dataset_yesno(
    model,
    processor,
    dataset,
    return_accuracy=False,
    progress_bar=True,
    progress_desc="Evaluating yes/no",
    batch_size=16,
):
    rows = []
    correct = 0
    total = 0

    indices = range(0, len(dataset), batch_size)
    iterator = tqdm(
        indices,
        total=(len(dataset) + batch_size - 1) // batch_size,
        desc=progress_desc,
    ) if progress_bar else indices

    for start_idx in iterator:
        end_idx = min(start_idx + batch_size, len(dataset))
        batch_items = [dataset[j] for j in range(start_idx, end_idx)]

        batch_scores = score_candidates_yes_fast(model, processor, batch_items)

        for item, scores in zip(batch_items, batch_scores):
            scores_arr = np.array(scores, dtype=np.float32)
            pred = int(np.argmax(scores_arr))

            if len(scores_arr) > 1:
                top2 = np.partition(scores_arr, -2)[-2:]
                top_margin = float(top2[-1] - top2[-2])
            else:
                top_margin = 999.0

            row = {
                "id": item["id"],
                "answer": pred,
                "scores": scores,
                "top_margin": top_margin,
            }

            if "answer" in item:
                gold = int(item["answer"])
                row["gold"] = gold
                total += 1
                if pred == gold:
                    correct += 1

            rows.append(row)

        if progress_bar and total > 0:
            iterator.set_postfix(running_acc=f"{correct / total:.4f}")

    pred_df = pd.DataFrame(rows)

    if return_accuracy and "gold" in pred_df.columns:
        acc = (pred_df["answer"] == pred_df["gold"]).mean()
        return pred_df, acc

    return pred_df


YES: [9230]  Yes
NO: [2838]  No


In [ ]:
val_pred_df, val_acc = predict_dataset_yesno(
    model=model,
    processor=processor,
    dataset=val_eval_ds,
    return_accuracy=True,
    progress_bar=True,
    progress_desc="Validating yes/no",
    batch_size=16,
)

print(f"Validation accuracy: {val_acc:.4f}")

print("\nPrediction distribution:")
print(val_pred_df["answer"].value_counts(normalize=True).sort_index())

print("\nGold distribution:")
print(val_pred_df["gold"].value_counts(normalize=True).sort_index())

print("\nMargin diagnostics:")
print("Mean top margin:", val_pred_df["top_margin"].mean())
print("Near-tie rate < 0.05:", (val_pred_df["top_margin"] < 0.05).mean())
print("Near-tie rate < 0.10:", (val_pred_df["top_margin"] < 0.10).mean())

val_pred_df.head()

In [21]:
# ── 13. Create submission.csv ───────────────────────────────────────────────
test_pred_df = predict_dataset_yesno(
    model=model,
    processor=processor,
    dataset=test_eval_ds,
    return_accuracy=False,
    progress_bar=True,
    progress_desc="Scoring test candidate yes/no",
    batch_size=8,
)

submission_df = test_pred_df[["id", "answer"]].copy()
submission_path = OUTPUT_DIR / "submission.csv"
submission_df.to_csv(submission_path, index=False)

print(f"Saved submission to: {submission_path}")
print("\nTest prediction distribution:")
print(submission_df["answer"].value_counts(normalize=True).sort_index())

display(submission_df.head())

Scoring test candidate yes/no:   0%|          | 0/126 [00:00<?, ?it/s]

Saved submission to: p2p_candidate_yesno_outputs_train_val/submission.csv

Test prediction distribution:
answer
0    0.359127
1    0.356151
2    0.204365
3    0.072421
4    0.007937
Name: proportion, dtype: float64


,id,answer
0,test_01750,3
1,test_00128,2
2,test_02891,0
3,test_02425,0
4,test_00930,4


In [22]:
# ── 14. Basic submission sanity checks ───────────────────────────────────────
sample_submission_path = DATA_DIR / "sample_submission.csv"

sub = pd.read_csv(OUTPUT_DIR / "submission.csv")
print(sub.head())
print("Rows:", len(sub))
print("Columns:", list(sub.columns))
assert list(sub.columns) == ["id", "answer"]

if sample_submission_path.exists():
    sample = pd.read_csv(sample_submission_path)
    assert set(sub["id"]) == set(sample["id"]), "Submission ids do not match sample_submission ids."
    print("IDs match sample_submission.csv.")

test_choices = dict(zip(test_df["id"], test_df["num_choices"]))
bad = []
for _, r in sub.iterrows():
    ans = int(r["answer"])
    if ans < 0 or ans >= int(test_choices[r["id"]]):
        bad.append((r["id"], ans, "out_of_range"))

print("Invalid rows:", len(bad))
assert len(bad) == 0, bad[:5]

           id  answer
0  test_01750       3
1  test_00128       2
2  test_02891       0
3  test_02425       0
4  test_00930       4
Rows: 1008
Columns: ['id', 'answer']
IDs match sample_submission.csv.
Invalid rows: 0
